# 🤟 Hand Gesture Detection using YOLOv8
**Artificial Intelligence 8.0 — Lab Activity**

---

This notebook implements a hand gesture detection system using the YOLOv8 algorithm.
The model is trained on a pre-annotated dataset from Roboflow Universe.

**Dataset:** Hand Gesture Recognition by Lebanese University  
**Dataset Link:** https://universe.roboflow.com/lebanese-university-grkoz/hand-gesture-recognition-y5827  
**Model:** YOLOv8n (pretrained on COCO, fine-tuned via transfer learning)  

**Gesture Classes:** one ☝️ · two ✌️ · three 🤟 · four 🖖 · five 🖐️

---

## ✅ Step 1 — Check GPU Runtime
Go to **Runtime → Change runtime type → T4 GPU → Save**

In [ ]:
import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
print(f"Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if not torch.cuda.is_available():
    print("\n⚠️  Go to Runtime → Change runtime type → T4 GPU → Save")
else:
    print("\n✅ GPU is ready!")

## ✅ Step 2 — Install Required Libraries

In [ ]:
!pip install ultralytics
!pip install roboflow

## ✅ Step 3 — Download Dataset from Roboflow

> 📌 Dataset: https://universe.roboflow.com/lebanese-university-grkoz/hand-gesture-recognition-y5827  
> 📌 Version: 5 | Images: 839 | Resolution: 416×416  
> 📌 Classes: one, two, three, four, five

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="nQJWtAPs4aV0dPlvdKV1")
project = rf.workspace("lebanese-university-grkoz").project("hand-gesture-recognition-y5827")
version = project.version(5)
dataset = version.download("yolov8")

print("\n✅ Dataset downloaded!")
print(f"Location: {dataset.location}")

## ✅ Step 4 — Verify Dataset Structure

In [ ]:
import os

DATASET_PATH = dataset.location

print("=== Folder Structure ===")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')

print("\n=== Image Count per Split ===")
for split in ["train", "valid", "test"]:
    split_path = os.path.join(DATASET_PATH, split, "images")
    if os.path.exists(split_path):
        count = len(os.listdir(split_path))
        print(f"  {split:>6} : {count} images")
    else:
        print(f"  {split:>6} : ⚠️ not found")

print("\n=== data.yaml ===")
with open(os.path.join(DATASET_PATH, "data.yaml"), "r") as f:
    print(f.read())

## ✅ Step 4.1 — Fix data.yaml Paths

In [ ]:
import yaml

yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(DATASET_PATH, "train/images")
data['val']   = os.path.join(DATASET_PATH, "valid/images")
data['test']  = os.path.join(DATASET_PATH, "test/images")

with open(yaml_path, "w") as f:
    yaml.dump(data, f)

print("✅ data.yaml fixed!")
print(f"  Classes ({data['nc']}): {data['names']}")
print(f"  train : {data['train']}")
print(f"  val   : {data['val']}")
print(f"  test  : {data['test']}")

## ✅ Step 5 — Load the Pretrained YOLOv8 Model

We load `yolov8n.pt` pretrained on COCO and fine-tune it on our hand gesture dataset using **transfer learning**.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("✅ YOLOv8n loaded!")

## ✅ Step 6 — Train the Model

Metrics recorded each epoch:
- `box_loss` — bounding box regression loss
- `cls_loss` — classification loss
- `dfl_loss` — distribution focal loss
- `Precision`, `Recall`, `mAP50`, `mAP50-95`

In [ ]:
results = model.train(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    epochs=100,
    imgsz=416,
    batch=16,
    name="hand_gesture_detector",
    patience=50,
    device=0
)

print("\n✅ Training complete!")

## ✅ Step 7 — Display Training Results

In [ ]:
from IPython.display import Image, display
import glob

run_dirs = sorted(glob.glob("/content/runs/detect/hand_gesture_detector*"))
RUN_DIR = run_dirs[-1]
print(f"Run directory: {RUN_DIR}")

print("\n📊 Training & Validation Curves:")
display(Image(filename=f"{RUN_DIR}/results.png", width=900))

In [ ]:
print("📊 Label Distribution:")
display(Image(filename=f"{RUN_DIR}/labels.jpg", width=800))

In [ ]:
print("📊 Confusion Matrix:")
display(Image(filename=f"{RUN_DIR}/confusion_matrix.png", width=800))

## ✅ Step 8 — Evaluate the Model

In [ ]:
best_model = YOLO(f"{RUN_DIR}/weights/best.pt")

metrics = best_model.val(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    imgsz=416
)

In [ ]:
print("\n" + "="*55)
print("       MODEL PERFORMANCE METRICS SUMMARY")
print("="*55)
print(f"  Precision  (P)   : {metrics.box.mp:.4f}")
print(f"  Recall     (R)   : {metrics.box.mr:.4f}")
print(f"  mAP @ 0.50       : {metrics.box.map50:.4f}")
print(f"  mAP @ 0.50:0.95  : {metrics.box.map:.4f}")
print("="*55)
print()
print("  Precision  — Of all detections made, how many were correct.")
print("  Recall     — Of all actual gestures, how many were found.")
print("  mAP50      — Mean Average Precision at IoU threshold 0.50.")
print("  mAP50-95   — Mean Average Precision at IoU 0.50 to 0.95.")

## ✅ Step 9 — Test on Dataset Images

Run detection on unseen test images from the dataset.

In [ ]:
TEST_IMAGES_PATH = os.path.join(DATASET_PATH, "test/images")

image_results = best_model.predict(
    source=TEST_IMAGES_PATH,
    save=True,
    conf=0.25,
    iou=0.45,
    imgsz=416,
    name="hand_gesture_image_test"
)

print(f"\n✅ Detection complete on {len(image_results)} image(s).")

In [ ]:
output_dir = "/content/runs/detect/hand_gesture_image_test"
output_images = glob.glob(f"{output_dir}/*.jpg") + glob.glob(f"{output_dir}/*.png")

print(f"Found {len(output_images)} output image(s).\n")
for img_path in output_images[:6]:
    print(f"🖼️  {os.path.basename(img_path)}")
    display(Image(filename=img_path, width=500))
    print()

In [ ]:
print("=" * 60)
print("         IMAGE DETECTION DETAILS")
print("=" * 60)

for i, result in enumerate(image_results[:6]):
    print(f"\n--- Image {i+1}: {os.path.basename(result.path)} ---")
    if len(result.boxes) == 0:
        print("  No detections found.")
    else:
        for box in result.boxes:
            cls_id       = int(box.cls[0])
            label        = result.names[cls_id]
            conf         = float(box.conf[0])
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            print(f"  Class        : {label}")
            print(f"  Confidence   : {conf:.2%}")
            print(f"  Bounding Box : ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
            print()

## ✅ Step 10 — Upload & Test Your Own Hand Gesture Image

Upload a photo of your own hand showing 1–5 fingers and see if the model detects it correctly!

> 💡 Tip: Take a clear photo with good lighting and a simple background.

In [ ]:
from google.colab import files
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("📁 Upload your hand gesture image (jpg, png, etc.)")
uploaded = files.upload()

IMG_PATH = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {IMG_PATH}")

In [ ]:
# Run detection on uploaded image
custom_results = best_model.predict(
    source=IMG_PATH,
    save=True,
    conf=0.25,
    iou=0.45,
    imgsz=416,
    name="hand_gesture_custom"
)

result = custom_results[0]

# Show side by side: original vs detected
original = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(annotated)
axes[1].set_title("Detection Output", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig("custom_detection_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Detection complete!")

In [ ]:
# Print detection details
CLASS_EMOJI = {
    "one":   "☝️",
    "two":   "✌️",
    "three": "🤟",
    "four":  "🖖",
    "five":  "🖐️",
}

print("=" * 50)
print("     YOUR HAND GESTURE DETECTION RESULT")
print("=" * 50)

if len(result.boxes) == 0:
    print("  ⚠️  No gesture detected.")
    print("  Try again with better lighting or a clearer hand position.")
else:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        label  = result.names[cls_id]
        conf   = float(box.conf[0])
        emoji  = CLASS_EMOJI.get(label.lower(), "🤚")
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        print(f"  Gesture      : {emoji}  {label.upper()}")
        print(f"  Confidence   : {conf:.2%}")
        print(f"  Bounding Box : ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
        print()

## ✅ Step 11 — Download Trained Model Weights

Download `best.pt` — needed for the **Streamlit web app** deployment on GitHub.

In [ ]:
from google.colab import files

BEST_PT = f"{RUN_DIR}/weights/best.pt"

if os.path.exists(BEST_PT):
    print("⬇️ Downloading best.pt...")
    files.download(BEST_PT)
    print("✅ Saved! Upload this to your GitHub repo.")
else:
    print("⚠️ best.pt not found. Make sure training completed.")

## ✅ Step 12 — Comparative Analysis

### Question 1: What is object detection?

Object detection is a computer vision task that enables a machine learning model to identify **what** objects are present in an image or video and **where** they are located. Unlike image classification which only labels the entire image, object detection outputs both a class label and a bounding box for each detected object. In this activity, the model detects hand gestures (one, two, three, four, five) and draws bounding boxes around each gesture found in the image.

### Question 2: How does YOLO perform object detection?

YOLO (You Only Look Once) performs object detection by analyzing the **entire image in a single forward pass** through the neural network. It divides the image into a grid and simultaneously predicts bounding boxes and class probabilities for each grid cell. This makes YOLO significantly faster than traditional methods that scan the image multiple times. YOLOv8 improves on earlier versions with a more efficient backbone, anchor-free detection head, and better accuracy — making it ideal for real-time hand gesture detection.

### Question 3: What is the role of a pre-annotated dataset?

A pre-annotated dataset provides images that already have **bounding boxes and class labels** assigned to each object of interest. In this activity, the Hand Gesture Recognition dataset from Roboflow Universe contains 839 images with 5 gesture classes already labeled (one, two, three, four, five). This saves significant time because we did not have to manually draw bounding boxes ourselves. Instead, we could focus on training, evaluating, and analyzing the model. The dataset is also properly split into training, validation, and test sets which is essential for fair model evaluation.

### Question 4: What do Precision, Recall, and mAP measure?

- **Precision** — measures how accurate the model's positive detections are. A high precision means that when the model says it detected a hand gesture, it is usually correct. Formula: TP / (TP + FP)

- **Recall** — measures how well the model finds all actual instances of a hand gesture. A high recall means the model misses very few real gestures. Formula: TP / (TP + FN)

- **mAP (Mean Average Precision)** — is the overall performance metric that averages precision across all gesture classes and IoU thresholds. mAP50 uses an IoU threshold of 0.50, while mAP50-95 averages across thresholds from 0.50 to 0.95. Higher mAP means better overall detection performance.

### Question 5: What challenges did you encounter during training?

1. **Dataset path issues** — The data.yaml file sometimes used relative paths that did not match the actual folder locations in Colab, requiring manual path correction before training could begin.

2. **Visual similarity between gestures** — Some hand gestures look similar to each other (e.g., four and five fingers), causing the model to occasionally confuse related classes.

3. **Lighting and background variation** — Images taken in different lighting conditions and backgrounds affected detection confidence on unseen test images.

4. **Dataset size** — With only 839 images across 5 classes, the model needed more epochs to converge properly compared to larger datasets.

### Question 6: How can object detection performance be improved?

1. **More training data** — Collecting more images per class with varied backgrounds, lighting, and hand sizes would improve generalization.

2. **Larger model variant** — Switching from YOLOv8n (nano) to YOLOv8s or YOLOv8m would increase model capacity for better accuracy.

3. **Data augmentation** — Applying augmentations such as random rotation, brightness shifts, and flipping helps the model generalize to real-world conditions.

4. **Hyperparameter tuning** — Adjusting learning rate, batch size, and confidence thresholds can further optimize detection accuracy.

5. **Deployment and testing** — Deploying the trained model as a web app on Streamlit enables real-world testing with user-uploaded images, which helps identify weaknesses and guide further improvement.

---
## 📋 Submission Checklist

- ✅ Source code (this `.ipynb` notebook)
- ✅ Screenshots of training results (loss curves, metrics per epoch)
- ✅ Screenshots of image detection outputs (bounding boxes, labels, confidence)
- ✅ Screenshots of custom hand gesture detection (Step 10)
- ✅ Downloaded `best.pt` model weights
- ✅ Recorded performance metrics (Precision, Recall, mAP50, mAP50-95)
- ✅ Comparative analysis (Questions 1–6 answered above)
- ✅ Dataset link: https://universe.roboflow.com/lebanese-university-grkoz/hand-gesture-recognition-y5827

---
*God Bless! — AI 8.0 Lab Activity | Deadline: May 25, 2026*